In [ ]:
import os
import shutil
import tkinter as tk
from tkinter import filedialog, messagebox
from ultralytics import YOLO

def select_root_folder():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    folder_selected = filedialog.askdirectory(title="탐색할 최상위 폴더를 선택하세요")
    root.destroy()
    return folder_selected

# --- 설정 ---
model_path = './yolov8n.pt'  
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

root_dir = select_root_folder()
if not root_dir:
    exit()

print("🤖 AI 모델 로딩 중...")
model = YOLO(model_path)

# --- 탐색 시작 ---
for root, dirs, files in os.walk(root_dir):
    # 'images', 'image', 'labels' 폴더 내부를 다시 탐색하지 않도록 방지
    current_folder_name = os.path.basename(root).lower()
    if current_folder_name in ['images', 'image', 'labels']:
        continue
    
    # 1. 대상 확인
    image_in_root = [f for f in files if f.lower().endswith(image_extensions)]
    existing_dirs = [d.lower() for d in dirs]
    
    target_images_dir = None
    
    # CASE 1: 'images' 또는 'image' 폴더가 이미 존재하는 경우
    if 'images' in existing_dirs:
        target_images_dir = os.path.join(root, 'images')
    elif 'image' in existing_dirs:
        target_images_dir = os.path.join(root, 'image')
    
    # CASE 2: 폴더는 없지만 사진 파일이 루트에 있는 경우 (images 폴더 생성 및 이동)
    elif image_in_root:
        target_images_dir = os.path.join(root, 'images')
        os.makedirs(target_images_dir, exist_ok=True)
        for img_name in image_in_root:
            shutil.move(os.path.join(root, img_name), os.path.join(target_images_dir, img_name))
        print(f"📦 폴더 생성 및 이동 완료: {target_images_dir}")

    # 처리가 필요한 폴더인 경우 라벨링 진행
    if target_images_dir:
        # labels 폴더는 항상 images/image 폴더와 같은 부모(root) 아래 위치
        target_labels_dir = os.path.join(root, 'labels')
        os.makedirs(target_labels_dir, exist_ok=True)
        
        process_files = [f for f in os.listdir(target_images_dir) if f.lower().endswith(image_extensions)]
        
        if not process_files:
            continue

        print(f"🚀 라벨링 작업 중: {target_images_dir}")
        
        for img_name in process_files:
            img_path = os.path.join(target_images_dir, img_name)
            file_base_name = os.path.splitext(img_name)[0]
            label_file_path = os.path.join(target_labels_dir, f"{file_base_name}.txt")

            # 이미 라벨링이 된 파일은 건너뛰기
            if os.path.exists(label_file_path):
                continue

            results = model.predict(source=img_path, conf=0.3, verbose=False)
            
            yolo_data = []
            for r in results:
                for box in r.boxes:
                    class_id = int(box.cls[0]) 
                    xywhn = box.xywhn[0].tolist()
                    line = f"{class_id} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}"
                    yolo_data.append(line)
            
            with open(label_file_path, 'w', encoding='utf-8') as f:
                f.write("\n".join(yolo_data))

print(f"\n✨ 모든 작업 완료!")
messagebox.showinfo("완료", "모든 하위 폴더 정리가 완료되었습니다.")

🤖 AI 모델 로딩 중...
🚀 'D:/DATA/highway_cctv/Preprocessed_Results' 탐색 시작...
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preprocessed_Results\0_origin\images
📦 이미지 이동 및 폴더 생성: D:/DATA/highway_cctv/Preprocessed_Results\0_origin\images -> /images
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preprocessed_Results\1_CLAHE\images
📦 이미지 이동 및 폴더 생성: D:/DATA/highway_cctv/Preprocessed_Results\1_CLAHE\images -> /images
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preprocessed_Results\2_Gaussian_Blur\images
📦 이미지 이동 및 폴더 생성: D:/DATA/highway_cctv/Preprocessed_Results\2_Gaussian_Blur\images -> /images
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preprocessed_Results\3_HSV_Fire_Mask\images
📦 이미지 이동 및 폴더 생성: D:/DATA/highway_cctv/Preprocessed_Results\3_HSV_Fire_Mask\images -> /images
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preprocessed_Results\4_Morphology_Closing\images
📦 이미지 이동 및 폴더 생성: D:/DATA/highway_cctv/Preprocessed_Results\4_Morphology_Closing\images -> /images
📂 기존 'images' 폴더 발견: D:/DATA/highway_cctv/Preproc